# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vedika1304-05/flyrank-internship-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [16]:
!pip install reportlab

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*
Lane chosen: Refresh / Content Opportunity Scoring


---


**Why this Lane?**
1. Numbers already show the scale: 93.8% of clients (30 of 32) have at least one declining page, and 54.2% of all 30,000 pages (16,262) are currently declining. No content team can manually inspect over half their inventory every week. There must be a model which automatically detects & ranks the pages requiring refresh/modification.
2. The question is about prioritization- "which pages should be reviewed first for refresh, expansion, protection, pruning, or monitoring?"
3. Its output is a ranked queue which can be used by the content team to work on, and decide which declining pages to prioritize. Output is deliverable that the real team can use.
4. A model that learns from multiple weighted signals at once (the way the starter random forest already does) captures relationships a single hand-written threshold structurally cannot.

**The decision and action this produces**
1. Who acts: a content strategist/editor with limited weekly capacity.
2. The action: review the top N pages in the queue first — refresh, expand, protect, prune, or monitor, per the reason code attached.
3. The cost of getting it wrong: wasted editor time on a false positive (low-value page reviewed for nothing), or a real, high-traffic decliner sitting unseen and continuing to lose visibility (a false negative).

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd, json
!git clone https://github.com/Vedika1304-05/flyrank-internship-ml.git
%cd flyrank-internship-ml
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# --- Claim 1: Is 54% decline real, and not just noise on dead pages? ---
n_total = len(df)
n_declining = (df["trend_direction"] == "down").sum()
pct_declining = n_declining / n_total * 100
visible_declining = df[(df["trend_direction"]=="down") & (df["impressions_90d"]>=100)]
print(f"Declining pages: {n_declining:,} ({pct_declining:.1f}%)")
print(f"Declining AND with real traffic: {len(visible_declining):,} "
      f"({len(visible_declining)/n_total*100:.1f}% of all pages)")

# --- Claim 2: Is the cost of a wrong call asymmetric (uneven)? ---
declining = df[df["trend_direction"]=="down"]
median_impr = declining["impressions_90d"].median()
high = declining[declining["impressions_90d"] >= median_impr]
low  = declining[declining["impressions_90d"] <  median_impr]
ratio = high["impressions_90d"].median() / max(low["impressions_90d"].median(), 1)
print(f"High-traffic decliners vs low-traffic decliners: {ratio:.1f}x more impressions")

# --- Claim 3: Does staleness alone explain decline? ---
med_days_declining = declining["days_since_last_update"].median()
med_days_all = df["days_since_last_update"].median()
print(f"Median days since update — declining: {med_days_declining:.0f} | all pages: {med_days_all:.0f}")

# --- Claim 4: Model vs. hand-written rule (from the trained pipeline) ---
!python scripts/run_all.py
res = json.load(open("outputs/model_results.json"))
base = res["baseline"]["baseline_precision_at_50"]
rf   = res["models"]["random_forest"]["precision_at_50"]
print(f"Baseline rule Precision@50: {base:.3f} | Random forest Precision@50: {rf:.3f} "
      f"-> {rf/base:.2f}x improvement")

Cloning into 'flyrank-internship-ml'...
remote: Enumerating objects: 150, done.
remote: Counting objects: 100% (150/150), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 150 (delta 58), reused 91 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (150/150), 1.92 MiB | 13.65 MiB/s, done.
Resolving deltas: 100% (58/58), done.
/content/flyrank-internship-ml/flyrank-internship-ml/flyrank-internship-ml/flyrank-internship-ml/flyrank-internship-ml/flyrank-internship-ml
Declining pages: 16,262 (54.2%)
Declining AND with real traffic: 13,152 (43.8% of all pages)
High-traffic decliners vs low-traffic decliners: 21.4x more impressions
Median days since update — declining: 20 | all pages: 20

▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-internship-ml/flyrank-internship-ml/flyrank-internship-ml/flyrank-internship-ml/flyrank-internship-ml/flyrank-internship-ml

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**decision improved:** which declining pages a content team should refresh first, given limited weekly refresh capacity.

**who acts on it:** a content strategist or SEO manager working through a prioritized queue — not an automated system. This is a decision-support tool, not an autonomous action.

**The unit of analysis:** one page (content_id) per row — the same grain as the raw data.

**cost of wrong recommendations:**
1. false positives: the content team wastes time in updating not so important pages, thereby leading to inefficient usage of time and resources.
2. false negatives: a genuinely high-opportunity page ranked low, so it never gets refreshed. The page keeps declining unnoticed, and the client silently loses search traffic/revenue over the following months.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

preds = pd.read_csv("data/processed/model_predictions.csv")
base  = pd.read_csv("data/processed/baseline_refresh_queue.csv")
final = pd.read_csv("outputs/refresh_queue.csv")

# Only score on the TEST split — clients/pages the model never trained on.
test_ids = set(preds.loc[preds["split"] == "test", "content_id"])
test_base = base[base["content_id"].isin(test_ids)].merge(
    preds[["content_id", "best_model_probability"]], on="content_id"
)
test_final = final[final["content_id"].isin(test_ids)]

CAPACITY = 50  # a content team can realistically review 50 pages/week

def evaluate_queue(df, score_col, capacity=CAPACITY):
    ranked = df.sort_values(score_col, ascending=False).reset_index(drop=True)
    reviewed, not_reviewed = ranked.head(capacity), ranked.iloc[capacity:]
    tp = reviewed["is_declining_label"].sum()
    total_declining_traffic = df.loc[df["is_declining_label"] == 1, "impressions_90d"].sum()
    missed_traffic = not_reviewed.loc[not_reviewed["is_declining_label"] == 1, "impressions_90d"].sum()
    return {
        "precision@50": round(tp / capacity, 3),
        "false_positives": int(capacity - tp),
        "pct_of_at_risk_traffic_caught": round((1 - missed_traffic / total_declining_traffic) * 100, 1),
    }

for name, df, col in [
    ("Baseline rule",       test_base,  "baseline_refresh_score"),
    ("Model only",          test_base,  "best_model_probability"),
    ("Final blend (70/30)", test_final, "final_refresh_score"),
]:
    print(f"{name:22s} -> {evaluate_queue(df, col)}")

high_value_decliners = declining[declining["impressions_90d"] >= declining["impressions_90d"].median()]
low_value_decliners = declining[declining["impressions_90d"] < declining["impressions_90d"].median()]

print(f"Declining pages ABOVE median traffic: {len(high_value_decliners):,}")
print(f"  → median impressions_90d: {high_value_decliners['impressions_90d'].median():,.0f}")
print(f"Declining pages BELOW median traffic: {len(low_value_decliners):,}")
print(f"  → median impressions_90d: {low_value_decliners['impressions_90d'].median():,.0f}")

# Spread in traffic size = spread in cost of getting priority wrong
print(f"\nRatio of high-value to low-value median traffic: "
      f"{high_value_decliners['impressions_90d'].median() / max(low_value_decliners['impressions_90d'].median(),1):.1f}x")

Baseline rule          -> {'precision@50': np.float64(0.24), 'false_positives': 38, 'pct_of_at_risk_traffic_caught': np.float64(15.4)}
Model only             -> {'precision@50': np.float64(0.84), 'false_positives': 8, 'pct_of_at_risk_traffic_caught': np.float64(5.5)}
Final blend (70/30)    -> {'precision@50': np.float64(0.82), 'false_positives': 9, 'pct_of_at_risk_traffic_caught': np.float64(17.2)}
Declining pages ABOVE median traffic: 8,133
  → median impressions_90d: 3,831
Declining pages BELOW median traffic: 8,129
  → median impressions_90d: 179

Ratio of high-value to low-value median traffic: 21.4x


The results show that even within the declining pages, the cost associated with missing a single page is fairly different (~21 times). So, missing a page from the upper-half will cause a greater loss/cost than missing one in the lower-half. This is exactly what the model misses to consider while ranking the pages.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# --- Number 1: SCALE — is this actually a big enough problem to matter? ---
declining = df[df["trend_direction"] == "down"]
visible_declining = declining[declining["impressions_90d"] >= 100]
pct_declining = len(declining) / len(df) * 100
print(f"1. SCALE: {len(declining):,} of {len(df):,} pages are declining ({pct_declining:.1f}%), "
      f"and {len(visible_declining):,} of those still have real traffic (≥100 impr/90d).")
print("   -> More than half the inventory needs review — this isn't a rare edge case.\n")

# --- Number 2: UNEVENNESS — does a flat yes/no label lose something important? ---
median_impr = declining["impressions_90d"].median()
high = declining[declining["impressions_90d"] >= median_impr]
low  = declining[declining["impressions_90d"] < median_impr]
ratio = high["impressions_90d"].median() / max(low["impressions_90d"].median(), 1)
print(f"2. UNEVEN STAKES: within declining pages, the top half has a {ratio:.1f}x higher "
      f"median traffic ({int(high['impressions_90d'].median()):,}) than the bottom half "
      f"({int(low['impressions_90d'].median()):,}).")
print("   -> Missing the wrong page costs vastly more than missing another — a ranked")
print("      score can reflect that; a flat 'declining: yes/no' flag cannot.\n")

# --- Number 3: THE MODEL EARNS ITS PLACE — a naive rule genuinely underperforms ---
med_days_declining = declining["days_since_last_update"].median()
med_days_all = df["days_since_last_update"].median()
print(f"3. NOT A TRIVIAL LOOKUP: median days-since-update is {med_days_declining:.0f} for "
      f"declining pages vs. {med_days_all:.0f} for all pages — essentially identical.")
print("   -> The obvious 'just flag stale pages' rule wouldn't work here, which is exactly")
print("      why this needs real modeling across multiple signals, not a single threshold.")

1. SCALE: 16,262 of 30,000 pages are declining (54.2%), and 13,152 of those still have real traffic (≥100 impr/90d).
   -> More than half the inventory needs review — this isn't a rare edge case.

2. UNEVEN STAKES: within declining pages, the top half has a 21.4x higher median traffic (3,831) than the bottom half (179).
   -> Missing the wrong page costs vastly more than missing another — a ranked
      score can reflect that; a flat 'declining: yes/no' flag cannot.

3. NOT A TRIVIAL LOOKUP: median days-since-update is 20 for declining pages vs. 20 for all pages — essentially identical.
   -> The obvious 'just flag stale pages' rule wouldn't work here, which is exactly
      why this needs real modeling across multiple signals, not a single threshold.


**Key findings:**
1. 16,262 of 30,000 pages (54.2%) are currently declining, matching the dictionary's stated label distribution exactly — confirming the data loaded correctly and that this is a large, non-niche population worth prioritizing.
2. Median days since last update is identical for declining pages and all pages (20 days each). This is a genuinely useful negative result. It means staleness alone does not obviously separate decliners from non-decliners in this sample, so my refresh-opportunity model can't rely on "how old is this content" as a standalone signal. This pushes me toward combining staleness with other signals (position trend, competition, content type) rather than treating recency as the primary driver

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:**
1. The output is just for 'decision-support' and the model by itself isn't taking any autonomous action.
2. The relationship between CTR, or position and decline is direct & real outcome of the data, and hence it is a real observation.
3. I can say the model outperforms the hand-written baseline — Precision@50 of 0.74–0.84 versus 0.24 — because that's a measured, reproducible result from a client-holdout test set, not a guess.
4. I can report that the model alone, despite higher precision, actually catches less at-risk traffic than the baseline (5.5% vs. 15.4%), and that blending the two closes that gap.

**What I can't claim:**
1. I can't say that refreshing a page causes recovery as I only have the past data and for justifying the above claim I will need the data after performing this experiment. Due to this limitation, I cannot verify if the recommendation actually works.
2. I will never claim to have identified a Google ranking factor or reverse-engineered the algorithm. My score reflects patterns in FlyRank's own observed data — traffic, clicks, position — not the hundreds of signals Google's actual system uses internally.
3. I cannot claim that - 'a declining page is guaranteed to be worth fixing'.




In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("CHECK 1 — No client names/URLs/product flags in the data (CAN claim: decision-support, not circular)")
print("="*70)
banned_cols = {"health_score", "priority_score", "action_type", "refresh_tier"}
present = banned_cols.intersection(df.columns)
print(f"Product decision-flag columns present in dataset: {present if present else 'NONE'}")
print("-> Nothing here to accidentally train on or copy from.\n")

id_cols = ["content_id", "client_id"]
for col in id_cols:
    sample = df[col].dropna().astype(str).head(3).tolist()
    print(f"{col} sample values: {sample}")
looks_like_url = df["content_id"].astype(str).str.contains("http", case=False).any()
print(f"Any content_id containing a raw URL? {looks_like_url}\n")
print("CHECK 2 — Decision-support framing: score reflects OBSERVED signals only")
print("="*70)
observed_signal_cols = ["impressions_90d", "clicks_90d", "sessions_90d", "avg_position",
                         "ctr", "engagement_rate", "trend_direction", "trend_pct"]
missing = [c for c in observed_signal_cols if c not in df.columns]
print(f"All expected observed-signal columns present? {'Yes' if not missing else missing}")
print("-> My score is built entirely from measured behavior (what happened),")
print("   never from Google's internal ranking logic (which isn't in this data at all).\n")

CHECK 1 — No client names/URLs/product flags in the data (CAN claim: decision-support, not circular)
Product decision-flag columns present in dataset: NONE
-> Nothing here to accidentally train on or copy from.

content_id sample values: ['content_304f48230142', 'content_a1fb4e703a9e', 'content_9aa793d4d895']
client_id sample values: ['client_f369cb89fc', 'client_4e07408562', 'client_7f2253d7e2']
Any content_id containing a raw URL? False

CHECK 2 — Decision-support framing: score reflects OBSERVED signals only
All expected observed-signal columns present? Yes
-> My score is built entirely from measured behavior (what happened),
   never from Google's internal ranking logic (which isn't in this data at all).



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.